In [0]:
CREATE OR REPLACE TABLE nyc_mobility.validation.weather_validation AS

WITH total_rows AS (
    SELECT COUNT(*) AS total_count
    FROM nyc_mobility.clean.weather_silver
),

validation_results AS (

-- Elevation
SELECT
    'elevation' AS column_name,
    COUNT(*) AS failed_rows
FROM nyc_mobility.clean.weather_silver
WHERE elevation IS NULL
   OR elevation < -500
   OR elevation > 9000

UNION ALL

-- Latitude
SELECT
    'latitude',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE latitude IS NULL
   OR latitude NOT BETWEEN -90 AND 90

UNION ALL

-- Longitude
SELECT
    'longitude',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE longitude IS NULL
   OR longitude NOT BETWEEN -180 AND 180

UNION ALL

-- Observation Time
SELECT
    'observation_time',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE observation_time IS NULL
   OR observation_time > CURRENT_TIMESTAMP()

UNION ALL

-- Precipitation
SELECT
    'precipitation',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE precipitation IS NULL
   OR precipitation < 0

UNION ALL

-- Rain
SELECT
    'rain',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE rain IS NULL
   OR rain < 0

UNION ALL

-- Snowfall
SELECT
    'snowfall',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE snowfall IS NULL
   OR snowfall < 0

UNION ALL

-- Temperature
SELECT
    'temperature_2m',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE temperature_2m IS NULL
   OR temperature_2m < -90
   OR temperature_2m > 60

UNION ALL

-- Timezone
SELECT
    'timezone',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE timezone IS NULL
   OR TRIM(timezone) = ''

UNION ALL

-- Weather Code
SELECT
    'weather_code',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE weather_code IS NULL
   OR weather_code NOT IN (
       0,1,2,3,
       45,48,
       51,53,55,
       56,57,
       61,63,65,
       66,67,
       71,73,75,
       77,
       80,81,82,
       85,86,
       95,96,99
   )

UNION ALL

-- Wind Speed
SELECT
    'wind_speed_10m',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE wind_speed_10m IS NULL
   OR wind_speed_10m < 0
   OR wind_speed_10m > 500

UNION ALL

-- Duplicate Grain Check
SELECT
    'duplicate_grain',
    COUNT(*)
FROM (
    SELECT
        latitude,
        longitude,
        observation_time,
        COUNT(*) AS cnt
    FROM nyc_mobility.clean.weather_silver
    GROUP BY
        latitude,
        longitude,
        observation_time
    HAVING COUNT(*) > 1
)
)

SELECT
    v.column_name,
    t.total_count,
    v.failed_rows,

    ROUND(
        100.0 * v.failed_rows / t.total_count,
        2
    ) AS failed_percentage,

    CASE
        WHEN v.failed_rows = 0 THEN 'PASS'
        WHEN (100.0 * v.failed_rows / t.total_count) < 5 THEN 'WARN'
        ELSE 'FAIL'
    END AS dq_status,

    CURRENT_TIMESTAMP() AS validation_timestamp

FROM validation_results v
CROSS JOIN total_rows t

ORDER BY
    CASE dq_status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
        ELSE 3
    END,
    failed_percentage DESC;